<a href="https://colab.research.google.com/github/BitanS2000/News-Sentiment-vs-Market-Volatility-Tracker/blob/main/News_Sentiment_vs_Market_Volatility_Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import yfinance as yf

In [ ]:
import finnhub
import datetime
import time
import pytz

In [ ]:
from transformers import pipeline

In [ ]:
ai_portfolio = {
    "NVDA": "Hardware (Chips)",
    "AMD": "Hardware (Chips)",
    "MSFT": "Software/Cloud",
    "GOOGL": "Software/Cloud",
    "TSM": "Infrastructure",
    "AVGO": "Infrastructure",
}

tickers = list(ai_portfolio.keys())

In [ ]:
print("Fetching stock data from Yahoo Finance...")
raw_data = yf.download(tickers, period="6mo", group_by="ticker")

In [ ]:
processed_frames = []

In [ ]:
for ticker in tickers:
    df_ticker = raw_data[ticker].copy()
    df_ticker["Ticker"] = ticker
    df_ticker["Category"] = ai_portfolio[ticker]

    df_ticker["Daily_Return"] = df_ticker["Close"].pct_change()

    df_ticker["Rolling_Volatility"] = (
        df_ticker["Daily_Return"].rolling(window=5).std()
    )

    df_ticker = df_ticker.reset_index()

    columns_to_keep = [
        "Date",
        "Ticker",
        "Category",
        "Close",
        "Volume",
        "Daily_Return",
        "Rolling_Volatility",
    ]
    processed_frames.append(df_ticker[columns_to_keep])

In [ ]:
master_stock_df = pd.concat(processed_frames, ignore_index=True)

master_stock_df = master_stock_df.dropna()

print("\n Stock Data Successfully Retrieved and Structured!")
master_stock_df.head(10)

In [ ]:
master_stock_df.info()

In [ ]:
master_stock_df.to_csv("ai_stock_data.csv", index=False)

In [ ]:
#!pip install finnhub-python pandas

In [ ]:
FINNHUB_API_KEY = "d92gop1r01qnksfej2vgd92gop1r01qnksfej300"
finnhub_client = finnhub.Client(api_key=FINNHUB_API_KEY)

tickers = ["NVDA", "AMD", "MSFT", "GOOGL", "TSM", "AVGO"]

end_date = datetime.date.today()
start_date = end_date - datetime.timedelta(days=180)

from_date = start_date.strftime("%Y-%m-%d")
to_date = end_date.strftime("%Y-%m-%d")

news_data = []

print(f"Fetching news from {from_date} to {to_date}...")

In [ ]:
for ticker in tickers:
    print(f"Pulling headlines for {ticker}...")
    try:
        articles = finnhub_client.company_news(ticker, _from=from_date, to=to_date)

        for article in articles:
            news_data.append({
                "Date": datetime.datetime.fromtimestamp(article["datetime"]).strftime('%Y-%m-%d'),
                "Ticker": ticker,
                "Headline": article["headline"],
                "Summary": article["summary"],
                "URL": article["url"]
            })

        time.sleep(2)

    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")

news_df = pd.DataFrame(news_data)

print("\n News Data Successfully Retrieved and Structured!")
news_df.info()
news_df.head(10)

In [ ]:
news_df.to_csv("ai_news_data.csv", index=False)

In [ ]:
#!pip install transformers torch

In [ ]:
sentiment_pipeline = pipeline("sentiment-analysis", model="ProsusAI/finbert")

def get_sentiment_score(text):

    try:
        result = sentiment_pipeline(text)[0]
        label = result['label']
        confidence = result['score']

        if label == 'positive':
            return confidence
        elif label == 'negative':
            return -confidence
        else:
            return 0.0
    except Exception as e:
        return 0.0

news_df['Sentiment_Score'] = news_df['Headline'].apply(get_sentiment_score)


In [ ]:
daily_sentiment_df = news_df.groupby(['Date', 'Ticker'])['Sentiment_Score'].mean().reset_index()
daily_sentiment_df.rename(columns={'Sentiment_Score': 'Daily_Avg_Sentiment'}, inplace=True)

# Convert 'Date' column in daily_sentiment_df to datetime to match master_stock_df
daily_sentiment_df['Date'] = pd.to_datetime(daily_sentiment_df['Date'])

# Corrected line: Use pd.merge() for joining based on keys
final_df = pd.merge(master_stock_df, daily_sentiment_df, on=['Date', 'Ticker'], how='left')

final_df['Daily_Avg_Sentiment'] = final_df['Daily_Avg_Sentiment'].fillna(0.0)

final_df.to_csv("AI_Sector_Master_Data.csv", index=False)

In [ ]:
final_df.tail(10)


In [ ]:
final_df.describe()

In [ ]:
# Create a filtered dataframe containing only active news days
active_news_df = final_df[final_df['Daily_Avg_Sentiment'] != 0.0]

print(f"Original trading days: {len(final_df)}")
print(f"Trading days with active news: {len(active_news_df)}")
print(active_news_df['Daily_Avg_Sentiment'].describe())

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

analyzable_df = final_df[final_df['Daily_Avg_Sentiment'] != 0.0]

correlation_matrix = analyzable_df[['Daily_Return', 'Rolling_Volatility', 'Daily_Avg_Sentiment']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('AI Sector: Sentiment vs. Market Metrics Correlation')
plt.show()

In [ ]:
# Create an intensity metric (ignoring if it's positive or negative)
analyzable_df['Sentiment_Intensity'] = analyzable_df['Daily_Avg_Sentiment'].abs()

intensity_corr = analyzable_df['Rolling_Volatility'].corr(analyzable_df['Sentiment_Intensity'])
print(f"Correlation with Sentiment Intensity: {intensity_corr:.2f}")

In [ ]:
import matplotlib.pyplot as plt

# 1. Grab ALL NVIDIA data so our timeline (X-axis) is properly sized
nvda_full_df = final_df[final_df['Ticker'] == 'TSM'].sort_values('Date')

# 2. Isolate only the days where actual news happened (for our red dots)
nvda_news_events = nvda_full_df[nvda_full_df['Daily_Avg_Sentiment'] != 0.0]

# 3. Create the plot
fig, ax1 = plt.subplots(figsize=(14, 7))

# Plot Daily Stock Returns (Left Axis - Line Chart is better for full timelines)
color = '#1f77b4'
ax1.set_xlabel('Date', fontweight='bold')
ax1.set_ylabel('Daily Stock Return', color=color, fontweight='bold')
ax1.plot(nvda_full_df['Date'], nvda_full_df['Daily_Return'], color=color, alpha=0.7, label='NVDA Daily Return')
ax1.tick_params(axis='y', labelcolor=color)

# Add a subtle grid and a zero-line to easily see positive vs negative stock days
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.grid(True, alpha=0.2)

# Create a second axis for Sentiment Score
ax2 = ax1.twinx()
color = '#d62728'
ax2.set_ylabel('News Sentiment Score (FinBERT)', color=color, fontweight='bold')

# Plot ONLY the active news days as scatter points (Red dots)
ax2.scatter(nvda_news_events['Date'], nvda_news_events['Daily_Avg_Sentiment'],
            color=color, s=80, edgecolors='black', zorder=5, label='News Event')

# Formatting
plt.title('TSM: Event Study - News Sentiment vs. Stock Movements', fontsize=14, fontweight='bold')
fig.autofmt_xdate() # Automatically rotates dates so they fit nicely
fig.tight_layout()
# Zoom in on the active news window (mid-May to July)
plt.xlim(pd.Timestamp('2026-05-10'), pd.Timestamp('2026-07-10'))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Isolate the target stock (TSM)
target_ticker = "TSM"
# Ensure the Date column is in datetime format for clean chronological plotting
final_df["Date"] = pd.to_datetime(final_df["Date"])
ticker_full_df = final_df[final_df["Ticker"] == target_ticker].sort_values(
    "Date"
)

# 2. Isolate only the days where actual news happened
ticker_news_events = ticker_full_df[
    ticker_full_df["Daily_Avg_Sentiment"] != 0.0
]

# 3. Create the plot
fig, ax1 = plt.subplots(figsize=(14, 7))

# Plot 5-Day Rolling Volatility (Left Axis - Line Chart)
color_vol = "#9467bd"  # Deep purple for risk/volatility
ax1.set_xlabel("Date", fontweight="bold")
ax1.set_ylabel(
    "5-Day Rolling Volatility (Market Risk)", color=color_vol, fontweight="bold"
)
ax1.plot(
    ticker_full_df["Date"],
    ticker_full_df["Rolling_Volatility"],
    color=color_vol,
    alpha=0.8,
    linewidth=2,
    label="Rolling Volatility",
)
ax1.tick_params(axis='y', labelcolor=color_vol)

# Add a subtle background grid
ax1.grid(True, alpha=0.2)

# Create a second axis for your Average Sentiment Score
ax2 = ax1.twinx()
color_sent = "#d62728"  # Crimson red for sentiment events
ax2.set_ylabel(
    "Daily Avg Sentiment Score (FinBERT)", color=color_sent, fontweight="bold"
)

# Plot the active news days as scatter points
ax2.scatter(
    ticker_news_events["Date"],
    ticker_news_events["Daily_Avg_Sentiment"],
    color=color_sent,
    s=80,
    edgecolors="black",
    zorder=5,
    label="News Event",
)

# Chart Title and Layout Adjustments
plt.title(
    f"{target_ticker}: Market Volatility Tracker vs. Daily Average News Sentiment",
    fontsize=14,
    fontweight="bold",
)
fig.autofmt_xdate()

# Zoom into the exact timeframe where your news events are concentrated
plt.xlim(pd.Timestamp("2026-05-10"), pd.Timestamp("2026-07-10"))

fig.tight_layout()
plt.show()